In this notebook, the bronze table is inspected, cleaned and standardized, in order to create the silver layer.

In [0]:
%sql

SELECT * FROM data_lakehouse_databricks.bronze.bronze_crm_prd_info LIMIT 10

Issues to solve:

- [ ]  Find duplicates
- [ ]  Validate string values: Check extra spaces, Identify abbreviations to normalize
- [ ]  Validate dates values: Check Data Type, check the format, handle missing values
- [ ]  Validate numeric values
- [ ]  Standardize business key IDs to ensure tables can be joined correctly.
- [ ]  Check the name of columns and table and make a plan how to rename them to something friendly.

Load bronze table in a spark dataframe to clean it.

In [0]:
# Load bronze table into a Spark DataFrame
df_bronze = spark.table("data_lakehouse_databricks.bronze.bronze_crm_prd_info")

print(f'Raw bronze table has {df_bronze.select("prd_id").count()} rows')

# Display the DataFrame
display(df_bronze)

### Find duplicates

In [0]:
df_bronze = df_bronze.dropDuplicates(['prd_id'])

print(f'Without duplicates, bronze table has {df_bronze.select("prd_id").count()} rows')

display(df_bronze)

### Validate string values: Check extra spaces, Identify abbreviations to normalize

In [0]:
from pyspark.sql.functions import trim, regexp_replace

# Standardize string values: trim extra spaces and remove 'XX-YY-' prefix from prd_key,
# then truncate to 7 characters
from pyspark.sql.functions import substring

df_bronze = df_bronze.withColumn(
    "prd_key",
    regexp_replace(trim(df_bronze["prd_key"]), "^[A-Z]{2}-[A-Z]{2}-", "")
)

# Trim all other string columns to remove extra spaces
for column in df_bronze.columns:
    if dict(df_bronze.dtypes)[column] == "string":
        df_bronze = df_bronze.withColumn(column, trim(df_bronze[column]))

#print(f'After string standardization, bronze table has {df_bronze.select("cst_id").count()} rows')

display(df_bronze)

### Validate dates values: Check Data Type, check the format, handle missing values

In [0]:
from pyspark.sql.functions import col, coalesce, to_date, lit, when, date_format

# Clean and standardize date columns to dd/MM/yyyy format
# Handle missing/null or wrong values by setting them to 01/01/2000

date_columns = [column for column in df_bronze.columns if 'date' in column.lower() or 'dt' in column.lower()]

for date_col in date_columns:
    df_bronze = df_bronze.withColumn(
        date_col,
        coalesce(
            to_date(col(date_col)),
            lit("2000-01-01")
        )
    )
    # Convert to dd/MM/yyyy string format
    df_bronze = df_bronze.withColumn(
        date_col,
        when(col(date_col).isNotNull(), 
             date_format(col(date_col), "dd/MM/yyyy")
        ).otherwise("01/01/2000")
    )

display(df_bronze)

### Validate categorical

In [0]:
from pyspark.sql.functions import col, when

# Replace null values with 'missing' for gender and marital_status
df_bronze = df_bronze.withColumn(
    "prd_line",
    when(col("prd_line").isNull(), "missing").otherwise(col("prd_line"))
)#.withColumn(
#    "prd_cost",
#    when(col("prd_cost").isNull(), 0).otherwise(col("prd_cost"))
#)

display(df_bronze)

### Validate numeric values

In [0]:
from pyspark.sql.functions import col

# Convert cst_id and cst_key to integer and remove null rows
df_bronze = df_bronze.withColumn("prd_cost", col("prd_cost").cast("int")) #\
                     #.withColumn("cst_key", col("cst_key").cast("int"))

# Remove rows where cst_id or cst_key is null
df_bronze = df_bronze.filter(col("prd_id").isNotNull() & col("prd_key").isNotNull())

print(f'After converting to int and removing nulls, bronze table has {df_bronze.select("prd_id").count()} rows')

display(df_bronze)

### Check the name of columns and table and make a plan how to rename them to something friendly.

In [0]:
# Rename columns to friendly names
df_bronze = df_bronze.withColumnRenamed("prd_id", "id") \
                     .withColumnRenamed("prd_key", "key") \
                     .withColumnRenamed("prd_nm", "name") \
                     .withColumnRenamed("prd_line", "line") \
                     .withColumnRenamed("prd_cost", "cost") \
                     .withColumnRenamed("prd_start_dt", "start_date") \
                     .withColumnRenamed("prd_end_dt", "end_date")

display(df_bronze)

### Save the dataframe to a silver table

In [0]:
df_bronze.write.format('delta').mode('overwrite').saveAsTable('data_lakehouse_databricks.silver.silver_product_info')


In [0]:
%sql
SELECT * FROM data_lakehouse_databricks.silver.silver_product_info